In [1]:
# .gitignore 파일 생성
with open(".gitignore", "w") as f:
    f.write("""
# 데이터 및 대용량 파일
images/
labels/
*.zip
*.tar.gz

# 모델 가중치 및 결과물
outputs/
checkpoint/

# 환경 변수 및 설정
.env
.ipynb_checkpoints/
    """)

In [2]:
!mkdir images labels outputs

---

## AI 허브에서 필요한 데이터셋 불러오기
- AI허브 API키
    - 	75828970-71E2-4557-9ED6-AE33951FC77E

- 데이터셋 요약: 해당 데이터셋은 4가지 장소, 8가지 카메라 구도로 수집된 데이터이며, 각각 센서, 동영상, 이미지 데이터 + 각 라벨링 데이터(JSON)으로 구성됨.
- 진행 방향: 먼저 이미지 데이터에 대한 라벨링 데이터를 가져오고, 병원, 예상 설치 각도를 정하여 1가지 각도의 이미지 데이터셋만을 불러오기.

### 1. 환경 설정 및 API 구성

In [3]:
import os
import json
import tarfile
import zipfile
import requests
from pathlib import Path
from pprint import pprint

# ============================================================
# AI허브 API 설정
# ============================================================
API_KEY = "75828970-71E2-4557-9ED6-AE33951FC77E"
DATASET_KEY = 71641  # 낙상사고 위험동작 영상-센서 쌍 데이터

# AI허브 API 엔드포인트 (aihubshell v0.6 기준)
BASE_URL = "https://api.aihub.or.kr"
DOWNLOAD_URL = f"{BASE_URL}/down/0.6/{DATASET_KEY}.do"

# 파일 키 (라벨링 데이터)
FILE_KEYS = {
    "TL": 531136,  # Training 라벨링데이터 (126 MB)
    "VL": 531138,  # Validation 라벨링데이터 (16 MB)
}

# 프로젝트 경로
PROJECT_DIR = Path.cwd()
LABELS_DIR = PROJECT_DIR / "labels"
DOWNLOAD_DIR = PROJECT_DIR / "downloads"

# 다운로드 폴더 생성
DOWNLOAD_DIR.mkdir(exist_ok=True)
LABELS_DIR.mkdir(exist_ok=True)

print(f"프로젝트 경로: {PROJECT_DIR}")
print(f"라벨 저장 경로: {LABELS_DIR}")
print(f"다운로드 임시 경로: {DOWNLOAD_DIR}")

프로젝트 경로: c:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM
라벨 저장 경로: c:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM\labels
다운로드 임시 경로: c:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM\downloads


### 2. 다운로드 및 압축 해제 함수 정의

AI허브 API는 데이터를 `.tar` 아카이브로 내려보내고, 그 안에 실제 `.zip` 파일이 존재합니다.  
**다운로드 → tar 해제 → zip 해제** 순서로 진행합니다.

In [4]:
def download_from_aihub(file_key, save_name="download.tar"):
    """
    AI허브 API를 통해 특정 파일을 다운로드합니다.
    
    Args:
        file_key (int): 다운로드할 파일의 filekey
        save_name (str): 저장할 tar 파일명
    
    Returns:
        Path: 다운로드된 tar 파일 경로
    """
    save_path = DOWNLOAD_DIR / save_name
    
    print(f"📥 다운로드 시작... (filekey: {file_key})")
    print(f"   URL: {DOWNLOAD_URL}?fileSn={file_key}")
    
    response = requests.get(
        DOWNLOAD_URL,
        headers={"apikey": API_KEY},
        params={"fileSn": file_key},
        stream=True
    )
    
    if response.status_code != 200:
        print(f"❌ 다운로드 실패! HTTP {response.status_code}")
        print(f"   응답: {response.text[:500]}")
        return None
    
    # 파일 크기 확인
    total_size = int(response.headers.get('content-length', 0))
    print(f"   파일 크기: {total_size / (1024*1024):.1f} MB")
    
    # 스트리밍 다운로드 (진행률 표시)
    downloaded = 0
    with open(save_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            downloaded += len(chunk)
            if total_size > 0:
                pct = (downloaded / total_size) * 100
                print(f"\r   진행률: {pct:.1f}% ({downloaded/(1024*1024):.1f}/{total_size/(1024*1024):.1f} MB)", end="")
    
    print(f"\n✅ 다운로드 완료: {save_path}")
    return save_path


def extract_tar_and_zip(tar_path, extract_to):
    """
    다운로드된 tar 파일을 해제하고, 내부의 zip(.part) 파일도 해제합니다.
    AI허브는 zip 파일을 .part0, .part1... 형태로 분할 저장하므로
    병합 → zip 해제 → 정리 순서로 진행합니다.
    """
    import shutil
    
    # 1단계: tar 해제
    print(f"📦 tar 파일 해제 중... → {DOWNLOAD_DIR}")
    with tarfile.open(tar_path, 'r') as tar:
        tar.extractall(path=DOWNLOAD_DIR)
        members = tar.getnames()
        print(f"   tar 내부 파일 목록:")
        for m in members:
            print(f"     - {m}")
    
    # 2단계: .part 파일 병합 → .zip 생성
    part_files = sorted(DOWNLOAD_DIR.rglob("*.zip.part*"))
    if part_files:
        # zip 파일명 추출 (TL.zip.part0 -> TL.zip)
        zip_name = part_files[0].name.rsplit('.part', 1)[0]
        zip_path = part_files[0].parent / zip_name
        
        print(f"\n🔗 {len(part_files)}개 part 파일 병합 중 → {zip_name}")
        with open(zip_path, 'wb') as outfile:
            for pf in part_files:
                with open(pf, 'rb') as infile:
                    shutil.copyfileobj(infile, outfile)
        print(f"   병합 완료: {zip_path.stat().st_size / (1024*1024):.1f} MB")
        
        # part 파일 삭제
        for pf in part_files:
            os.remove(pf)
    
    # 3단계: zip 파일 해제
    for zip_file in DOWNLOAD_DIR.rglob("*.zip"):
        print(f"\n📦 zip 파일 해제 중: {zip_file.name} → {extract_to}")
        with zipfile.ZipFile(zip_file, 'r') as zf:
            zf.extractall(path=extract_to)
            file_count = len(zf.namelist())
            print(f"   ✅ {file_count}개 파일 추출 완료")
    
    # 4단계: 임시 파일 정리
    os.remove(tar_path)
    print(f"\n🗑️ 임시 tar 파일 삭제 완료")

### 3. Training 라벨링 데이터 다운로드 (TL.zip, 126MB)

In [5]:
# Training 라벨링 데이터 다운로드
tar_path = download_from_aihub(FILE_KEYS["TL"], save_name="TL_download.tar")

if tar_path:
    extract_tar_and_zip(tar_path, LABELS_DIR)

📥 다운로드 시작... (filekey: 531136)
   URL: https://api.aihub.or.kr/down/0.6/71641.do?fileSn=531136
   파일 크기: 0.0 MB

✅ 다운로드 완료: c:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM\downloads\TL_download.tar
📦 tar 파일 해제 중... → c:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM\downloads
   tar 내부 파일 목록:
     - 041.낙상사고_위험동작_영상-센서_쌍_데이터/3.개방데이터/1.데이터/Training/02.라벨링데이터/TL.zip.part0

🗑️ 임시 tar 파일 삭제 완료


### 4. 라벨링 데이터 구조 탐색

In [6]:
# 라벨링 폴더 내 파일/폴더 구조 확인
print("📂 라벨링 데이터 폴더 구조:")
print("=" * 50)

for root, dirs, files in os.walk(LABELS_DIR):
    level = root.replace(str(LABELS_DIR), '').count(os.sep)
    indent = '  ' * level
    folder_name = os.path.basename(root)
    file_count = len(files)
    print(f"{indent}📁 {folder_name}/ ({file_count}개 파일)")
    
    # 최대 5개 파일만 표시
    sub_indent = '  ' * (level + 1)
    for f in files[:5]:
        print(f"{sub_indent}📄 {f}")
    if file_count > 5:
        print(f"{sub_indent}... 외 {file_count - 5}개")

📂 라벨링 데이터 폴더 구조:
📁 labels/ (0개 파일)


In [7]:
# JSON 파일 하나를 열어서 구조 확인
json_files = list(LABELS_DIR.rglob("*.json"))
print(f"총 JSON 파일 수: {len(json_files)}개")
print("=" * 50)

if json_files:
    # 첫 번째 JSON 파일 상세 출력
    sample_file = json_files[0]
    print(f"\n📋 샘플 JSON 파일: {sample_file.name}")
    print("-" * 50)
    
    with open(sample_file, 'r', encoding='utf-8') as f:
        sample_data = json.load(f)
    
    print("\n🔑 최상위 키:")
    for key in sample_data.keys():
        print(f"  - {key}: {type(sample_data[key]).__name__}")
    
    print("\n📄 전체 JSON 내용:")
    pprint(sample_data, width=100, depth=3)
else:
    print("⚠️ JSON 파일을 찾을 수 없습니다.")

총 JSON 파일 수: 0개
⚠️ JSON 파일을 찾을 수 없습니다.


### 5. 주요 필드 분포 분석

촬영 장소, 카메라 번호, 낙상 여부/유형의 분포를 확인하여 병원 + 1개 카메라 구도 데이터를 선별합니다.

In [8]:
# 여러 JSON 파일을 읽어서 주요 필드값 분포 확인
import collections

# 최대 100개 파일 샘플링하여 분포 확인
sample_size = min(100, len(json_files))
sample_jsons = json_files[:sample_size]

# 주요 필드 수집
scene_locs = []      # 촬영 장소
cam_nums = []         # 카메라 번호
fall_types = []       # 낙상 유형
is_falls = []         # 낙상 여부

for jf in sample_jsons:
    try:
        with open(jf, 'r', encoding='utf-8') as f:
            data = json.load(f)
        
        # scene_info에서 필드 추출
        if 'scene_info' in data:
            info = data['scene_info']
            scene_locs.append(info.get('scene_loc', 'unknown'))
            cam_nums.append(str(info.get('cam_num', 'unknown')))
            fall_types.append(info.get('scene_cat_name', 'unknown'))
            is_falls.append(info.get('scene_IsFall', 'unknown'))
    except Exception as e:
        print(f"⚠️ {jf.name} 읽기 실패: {e}")

print(f"분석한 파일 수: {sample_size}개")
print("=" * 50)

print("\n📍 촬영 장소 분포:")
for loc, cnt in collections.Counter(scene_locs).most_common():
    print(f"  {loc}: {cnt}개")

print("\n📷 카메라 번호 분포:")
for cam, cnt in collections.Counter(cam_nums).most_common():
    print(f"  C{cam}: {cnt}개")

print("\n⚠️ 낙상 여부 분포:")
for fall, cnt in collections.Counter(is_falls).most_common():
    print(f"  {fall}: {cnt}개")

print("\n📊 낙상 유형 분포:")
for ft, cnt in collections.Counter(fall_types).most_common():
    print(f"  {ft}: {cnt}개")

분석한 파일 수: 0개

📍 촬영 장소 분포:

📷 카메라 번호 분포:

⚠️ 낙상 여부 분포:

📊 낙상 유형 분포:
